In [1]:

from pathlib import Path
import pandas as pd

# Ubicación de los archivos procesados
carpeta_datos = Path("../data/processed")

peliculas = pd.read_csv(
    carpeta_datos / "peliculas_procesadas.csv"
)

series = pd.read_csv(
    carpeta_datos / "series_procesadas.csv"
)

# Comprobar los datos cargados
print("=== PELÍCULAS ===")
print("Registros:", len(peliculas))
print("Columnas:", len(peliculas.columns))

print("\n=== SERIES ===")
print("Registros:", len(series))
print("Columnas:", len(series.columns))

print("\n=== COLUMNAS DISPONIBLES ===")
print("Películas:", peliculas.columns.tolist())
print("Series:", series.columns.tolist())


=== PELÍCULAS ===
Registros: 16000
Columnas: 18

=== SERIES ===
Registros: 15991
Columnas: 16

=== COLUMNAS DISPONIBLES ===
Películas: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
Series: ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'description', 'popularity', 'vote_count', 'vote_average']


In [2]:

# Copias de trabajo: los archivos CSV originales no se modifican
peliculas_analisis = peliculas.copy()
series_analisis = series.copy()

# 1. Identificar títulos con votos registrados
peliculas_analisis["has_votes"] = peliculas_analisis["vote_count"] > 0
series_analisis["has_votes"] = series_analisis["vote_count"] > 0

# 2. Crear valoraciones válidas para el análisis
# Si no hay votos, no usaremos la valoración en los promedios
peliculas_analisis["vote_average_analysis"] = (
    peliculas_analisis["vote_average"].where(
        peliculas_analisis["vote_count"] > 0
    )
)

series_analisis["vote_average_analysis"] = (
    series_analisis["vote_average"].where(
        series_analisis["vote_count"] > 0
    )
)

# 3. Calcular el indicador financiero definido en el caso
# Solo se calcula cuando presupuesto e ingresos son positivos
peliculas_analisis["revenue_budget_ratio"] = (
    peliculas_analisis["revenue"]
    .div(peliculas_analisis["budget"].where(
        (peliculas_analisis["budget"] > 0)
        & (peliculas_analisis["revenue"] > 0)
    ))
)

# 4. Resumen de comprobación
print("=== PELÍCULAS ===")
print("Títulos con votos:", peliculas_analisis["has_votes"].sum())
print(
    "Valoraciones válidas:",
    peliculas_analisis["vote_average_analysis"].notna().sum()
)
print(
    "Películas con indicador financiero:",
    peliculas_analisis["revenue_budget_ratio"].notna().sum()
)

print("\n=== SERIES ===")
print("Títulos con votos:", series_analisis["has_votes"].sum())
print(
    "Valoraciones válidas:",
    series_analisis["vote_average_analysis"].notna().sum()
)

print("\n=== EJEMPLO DEL INDICADOR FINANCIERO ===")
print(
    peliculas_analisis[
        ["title", "budget", "revenue", "revenue_budget_ratio"]
    ].dropna(subset=["revenue_budget_ratio"]).head()
)


=== PELÍCULAS ===
Títulos con votos: 15106
Valoraciones válidas: 15106
Películas con indicador financiero: 3540

=== SERIES ===
Títulos con votos: 12325
Valoraciones válidas: 12325

=== EJEMPLO DEL INDICADOR FINANCIERO ===
                                          title     budget    revenue  \
0                           Shrek Forever After  165000000  752600867   
1                                     Inception  160000000  839030630   
2  Harry Potter and the Deathly Hallows: Part 1  250000000  954305868   
3                                       Tangled  260000000  592461732   
4                      How to Train Your Dragon  165000000  494879471   

   revenue_budget_ratio  
0              4.561217  
1              5.243941  
2              3.817223  
3              2.278699  
4              2.999270  


In [3]:

from pathlib import Path

carpeta_procesados = Path("../data/processed")
carpeta_procesados.mkdir(parents=True, exist_ok=True)

ruta_peliculas = carpeta_procesados / "peliculas_analisis.csv"
ruta_series = carpeta_procesados / "series_analisis.csv"

peliculas_analisis.to_csv(ruta_peliculas, index=False)
series_analisis.to_csv(ruta_series, index=False)

print("=== ARCHIVOS DE ANÁLISIS GUARDADOS ===")
print("Películas:", ruta_peliculas)
print("Series:", ruta_series)

print("\nDimensiones:")
print("Películas:", peliculas_analisis.shape)
print("Series:", series_analisis.shape)

print("\nIndicadores financieros disponibles:",
      peliculas_analisis["revenue_budget_ratio"].notna().sum())


=== ARCHIVOS DE ANÁLISIS GUARDADOS ===
Películas: ..\data\processed\peliculas_analisis.csv
Series: ..\data\processed\series_analisis.csv

Dimensiones:
Películas: (16000, 21)
Series: (15991, 18)

Indicadores financieros disponibles: 3540
